In [1]:
import sys
import json
import pandas as pd
from pathlib import Path

root_dir = Path.cwd().parent

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.vector_store import initialize_vector_store
from src.llm_agent import decision_agent, rag_decision_agent, agentic_decision_system
from src.evaluator import (
    llm_judge,
    semantic_evaluator,
    risk_band,
    threshold_action,
    consistency_test
)

case_path = root_dir / "data" / "cases" / "cases.json"

with open(case_path, "r") as f:
    cases = json.load(f)

print(f"Loaded {len(cases)} cases.")

Loaded 200 cases.


In [6]:
from src.vector_store import initialize_vector_store

collection = initialize_vector_store(reset=True)
print("Vector store initialized.")

Vector store initialized.


In [7]:
#baseline agent vs RAG agent
comparison_rows = []

for case in cases[:5]:
    baseline_result = decision_agent(case)
    rag_result = rag_decision_agent(case)

    comparison_rows.append({
        "case_id": case["case_id"],
        "domain": case["domain"],
        "ground_truth": case["ground_truth_decision"],
        "baseline_decision": baseline_result.get("decision"),
        "baseline_risk": baseline_result.get("risk_score"),
        "rag_decision": rag_result.get("decision"),
        "rag_risk": rag_result.get("risk_score"),
        "retrieved_rules": rag_result.get("retrieved_rules")
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,case_id,domain,ground_truth,baseline_decision,baseline_risk,rag_decision,rag_risk,retrieved_rules
0,E-000,ecommerce,approve,escalate,0.7,approve,0.2,"[E-Commerce Return and Refund Policy, 5. Final..."
1,E-001,ecommerce,reject,reject,0.7,reject,0.7,"[E-Commerce Return and Refund Policy, 5. Final..."
2,E-002,ecommerce,approve,approve,0.2,approve,0.2,"[E-Commerce Return and Refund Policy, 5. Final..."
3,E-003,ecommerce,escalate,escalate,0.7,escalate,0.7,"[E-Commerce Return and Refund Policy, 6. Refun..."
4,E-004,ecommerce,escalate,escalate,0.7,escalate,0.5,"[E-Commerce Return and Refund Policy, 6. Refun..."


In [8]:
#evaluation
eval_rows = []

for case in cases[:5]:
    ai_result = rag_decision_agent(case)
    judge = llm_judge(case, ai_result)
    sem = semantic_evaluator(case, ai_result)

    eval_rows.append({
        "case_id": case["case_id"],
        "domain": case["domain"],
        "ground_truth": case["ground_truth_decision"],
        "ai_decision": ai_result.get("decision"),
        "risk_score": ai_result.get("risk_score"),
        "risk_band": risk_band(ai_result.get("risk_score")),
        "threshold_action": threshold_action(ai_result.get("risk_score")),
        "judge_score": judge.get("overall_score"),
        "reasoning_quality": judge.get("reasoning_quality"),
        "semantic_similarity": sem.get("semantic_similarity")
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

,case_id,domain,ground_truth,ai_decision,risk_score,risk_band,threshold_action,judge_score,reasoning_quality,semantic_similarity
0,E-000,ecommerce,approve,approve,0.2,low,auto_approve_allowed,1,1,0.654180
1,E-001,ecommerce,reject,reject,0.7,high,human_review_required,1,1,0.639168
2,E-002,ecommerce,approve,approve,0.2,low,auto_approve_allowed,1,1,0.697177
3,E-003,ecommerce,escalate,escalate,0.7,high,human_review_required,1,1,0.631687
4,E-004,ecommerce,escalate,escalate,0.6,medium,secondary_review,1,1,0.775192


In [9]:
#consistency test
consistency_rows = []

for case in cases[:5]:
    consistency_rows.append(
        consistency_test(case, rag_decision_agent, runs=5)
    )

consistency_df = pd.DataFrame(consistency_rows)
consistency_df

,case_id,decisions,scores,decision_consistent,score_variance
0,E-000,"[approve, approve, approve, approve, approve]","[0.2, 0.2, 0.2, 0.2, 0.2]",True,0.0
1,E-001,"[reject, reject, reject, reject, reject]","[0.7, 0.7, 0.7, 0.7, 0.6]",True,0.1
2,E-002,"[approve, approve, approve, approve, approve]","[0.2, 0.2, 0.2, 0.2, 0.2]",True,0.0
3,E-003,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.7, 0.6, 0.7, 0.7]",True,0.1
4,E-004,"[escalate, escalate, escalate, escalate, escal...","[0.7, 0.6, 0.6, 0.6, 0.7]",True,0.1


In [10]:
#Agentic workflow
test_case = cases[0]

agentic_result = agentic_decision_system(test_case)
agentic_result

{'decision': 'approve',
 'risk_score': 0.2,
 'risk_signals': ['high-value transaction', 'new account'],
 'policy_evidence': ['Final Sale Items', 'Refund Abuse Indicators'],
 'reasoning': 'The refund request is valid as the customer provided evidence for receiving the wrong item, and there are no indicators of refund abuse despite the high-value transaction.',
 'recommended_action': 'Process the refund and notify the customer.',
 'draft_summary': "The initial decision approved the refund based on the absence of final sale item classification and the customer's evidence.",
 'critique_summary': 'The critique highlighted unsupported claims regarding item classification, missing policy evidence for high-value transactions, weak reasoning on evidence relevance, and lack of explicit policy compliance assessment.',
 'retrieved_rules': ['E-Commerce Return and Refund Policy',
  '5. Final Sale Items\nFinal sale items are not eligible for return unless required by law.',
  '6. Refund Abuse Indicat

In [11]:
#agentic result as judge
agentic_judge = llm_judge(test_case, agentic_result)
agentic_semantic = semantic_evaluator(test_case, agentic_result)

{
    "case_id": test_case["case_id"],
    "decision": agentic_result.get("decision"),
    "risk_score": agentic_result.get("risk_score"),
    "judge_score": agentic_judge.get("overall_score"),
    "semantic_similarity": agentic_semantic.get("semantic_similarity")
}

{'case_id': 'E-000',
 'decision': 'approve',
 'risk_score': 0.2,
 'judge_score': 0,
 'semantic_similarity': 0.6497921797050016}

## Final Analysis

This notebook demonstrates a modular AI risk decision system.

### Components

1. Baseline LLM decision agent  
2. RAG-based policy retrieval agent  
3. LLM-as-Judge evaluator  
4. Embedding-based semantic reasoning evaluator  
5. Consistency test  
6. Risk calibration and thresholding  
7. Draft–critique–refine agentic workflow  

### Key Findings

- RAG enables the system to retrieve only relevant policy rules instead of passing full policy documents.
- LLM-as-Judge evaluates correctness, policy grounding, and reasoning quality.
- Semantic similarity provides an additional signal for reasoning alignment.
- Consistency testing shows whether decisions remain stable across repeated runs.
- Thresholding translates raw risk scores into operational review actions.
- Agentic draft–critique–refine workflow improves explainability but increases latency and API cost.

### Product Framing

This system is a prototype AI risk decision copilot for customer-facing digital transactions.  
It is designed for human-in-the-loop decision support rather than fully automated rejection or approval.